# 02. Opacus Fundamentals - Model Compatibility and ModuleValidator

As we move toward applying Differential Privacy to complex medical imaging or sequence models, we will encounter a major hurdle: **Not all standard PyTorch layers are compatible with DP-SGD.**

The biggest offender is **`BatchNorm`!**
<br><br>

### Why 'BatchNorm' Breaks Privacy?

`BatchNorm` computes the mean and variance across the _entire batch_ of data. DP-SGD requires isolating the gradients for each _individual sample_ to clip them and add noise. Because `BatchNorm` inherently mixes data from different samples in the forward pass, it makes calculating exact per-sample gradients impossible.

In this notebook, we will:
1. Build a model with incompatible layers.
2. Use Opacus's `ModuleValidator` to detect these layers.
3. Automatically rewrite the model to use privacy-compliant layers (e.g., `GroupNorm`).

## 1. Building a CNN Model Architecture

In this section, we define a simple CNN designed for small medical‑image classification tasks. The model uses a single convolutional block followed by pooling and a fully connected layer to produce two‑class predictions. This lightweight architecture is easy to train and serves as a clear starting point for differential‑privacy‑compatible design choices.

In [1]:
import torch
import torch.nn as nn
from opacus.validators import ModuleValidator

# 1. Create a standard CNN suitable for images (e.g., medical scans)
# !! Notice the use of BatchNorm2D

class MedicalCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16) # INCOMPITABLE WITH DP
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(16 * 14 * 14, 2)
        
    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x
    
model = MedicalCNN()
print(f"Original Model Architecture:\n\n{model}")

Original Model Architecture:

MedicalCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=3136, out_features=2, bias=True)
)


## 2. Vaidating the Model

If we tried to pass this model into `privacy_engine.make_private()`, Opacus would throw a loud error. We can proactively check for compatability using `ModuleValidator.is_valid()` and find out exactly what is wrong using `ModuleValidator.validate()`.

In [2]:
is_valid =  ModuleValidator.is_valid(model)
print(f"Is the model DP Compatible? --> {is_valid}\n")

# Let's see exactly which modules are violating the rules
errors = ModuleValidator.validate(model, strict=False)
print("Validation Errors:")
for error in errors:
    print(f"\t- {error}")

Is the model DP Compatible? --> False

Validation Errors:
	- BatchNorm cannot support training with differential privacy. The reason for it is that BatchNorm makes each sample's normalized value depend on its peers in a batch, ie the same sample x will get normalized to a different value depending on who else is on its batch. Privacy-wise, this means that we would have to put a privacy mechanism there too. While it can in principle be done, there are now multiple normalization layers that do not have this issue: LayerNorm, InstanceNorm and their generalization GroupNorm are all privacy-safe since they don't have this property.We offer utilities to automatically replace BatchNorms to GroupNorms and we will release pretrained models to help transition, such as GN-ResNet ie a ResNet using GroupNorm, pretrained on ImageNet


#### 🔍 Understanding the Compatibility Error

The `ModuleValidator` successfully caught a critical privacy violation in our model: **BatchNorm2d**. 

**Why does BatchNorm break Differential Privacy?**
In standard neural network training, `BatchNorm` normalizes data by calculating the mean and variance across an *entire batch* of inputs. This means the mathematical calculations for Patient A's data are inherently tangled with the data from Patient B, C, and D in the same batch. 

Because DP-SGD requires the `PrivacyEngine` to isolate, measure, and clip the gradient of *each individual sample independently*, this cross-sample mixing makes it impossible to apply per-sample gradient clipping. If the engine can't isolate a sample, it can't guarantee its privacy.

**The Privacy-Safe Solution: GroupNorm**
The Opacus error message explains that normalization layers like `LayerNorm`, `InstanceNorm`, and `GroupNorm` are perfectly safe. This is because they normalize data across the channels or spatial dimensions of a *single* input independently, rather than across multiple inputs in a batch. When we run the automatic fix in the next step, Opacus will automatically swap out our `BatchNorm` for a DP-compliant `GroupNorm` layer, preserving the model's ability to learn while maintaining strict mathematical isolation.

## 3. Fixing the Model Automatically

Instead of manually rewriting large model architectures (like a ResNet or a custom Transformer), Opacus provides a built-in fix() method.

This function traverses the model graph and replaces know incompatible layers with mathematically similar, DP-compliant equivalents. For example, it replcaes `BatchNorm` with `GroupNorm`.

In [3]:
# automatically replace incompatible layers
model_fixed = ModuleValidator.fix(model)

print(f"Fixed Model Architecture:\n\n{model_fixed}")

# re-validate to ensure it worked
print(f"\nIs the fixed model DP compatible? ---> {ModuleValidator.is_valid(model_fixed)}")

Fixed Model Architecture:

MedicalCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): GroupNorm(16, 16, eps=1e-05, affine=True)
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=3136, out_features=2, bias=True)
)

Is the fixed model DP compatible? ---> True


### 🎉 Summary
You have successfully learned how to audit standard PyTorch architectures and refactor them for DP-SGD. 

When working on complex projects where we might be pulling pre-trained base models from TorchVision or Hugging Face, running `ModuleValidator.fix(model)` will be our crucial first step before attaching the `PrivacyEngine`.

In the next notebook, we will move to training an end-to-end Image Classification model, putting everything we've learned together.